# FT-900 半导体出口 Exhibit 7 的分类边界

读取已归档并已核 NAS 的 188 个月报原始 PDF，核对每份 SHA/尺寸、报告内当月与同版上月两列，以及首页五页分类通知。2011-12、2012-01、2012-02 的通知和表格另作视觉检查。此卡不读取 SOXX 市场结果、不构造 Qlib 因子或拟合。

In [1]:
import json,hashlib,re,calendar
from pathlib import Path
import fitz
root=Path.cwd();card=root/'research/census_ft900_semiconductor_classification_boundary_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda f:hashlib.sha256(Path(f).read_bytes()).hexdigest()
src=root/plan['source']['card']
for file,key in [('summary.json','source_summary_sha256'),('full_training_row_audit.json','training_row_audit_sha256'),('held_row_identity_audit.json','held_row_audit_sha256')]:assert sha(src/file)==plan['source'][key]
summary=json.loads((src/'summary.json').read_text());assert summary['full_training_scoped_restores_verified']==12 and summary['held_scoped_restores_verified']==4
training=json.loads((src/'full_training_row_audit.json').read_text())['rows'];held=json.loads((src/'held_row_identity_audit.json').read_text())['rows']
assert len(training)==144 and len(held)==44
rows=sorted([(x,'training' if x in training else 'held') for x in training+held],key=lambda a:a[0]['reference_month'])
assert len({x['reference_month'] for x,_ in rows})==188
patterns=['Changes to End-Use Commodity Classification','semiconductor media','reassigned','Harmonized System']
notice_hits=[];values=[];boundary_pages={}
for x,role in rows:
 month=x['reference_month'];name='ft900_'+month[2:4]+month[5:7]+'.pdf';raw=root/plan['source'][role+'_raw_root']/name
 assert raw.exists() and raw.stat().st_size==x['source_bytes'] and sha(raw)==x['source_pdf_sha256'],month
 current=x['current_month_million_usd'];prior=x['prior_month_same_release_million_usd'];assert current>0 and prior>0
 values.append({'month':month,'current':current,'same_release_prior':prior})
 doc=fitz.open(raw)
 for pi in range(min(5,len(doc))):
  text=' '.join(doc[pi].get_text().split())
  if month in ('2011-12','2012-01','2012-02') and pi in ((3,) if month=='2011-12' else (2,)):
   boundary_pages[month]=text
  for pat in patterns:
   for match in list(re.finditer(re.escape(pat),text,re.I))[:2]:
    notice_hits.append({'month':month,'PDF_sha256':x['source_pdf_sha256'],'physical_page':pi+1,'phrase':pat,'context':text[max(0,match.start()-120):match.end()+260]})
assert len(boundary_pages)==3
assert 'semiconductor media' in boundary_pages['2012-02'].lower() and 'effective with the january 2012 statistics' in boundary_pages['2012-02'].lower()
assert 'Changes to End-Use Commodity Category Descriptions' in boundary_pages['2011-12'] and 'Changes to End-Use Commodity Category Descriptions' in boundary_pages['2012-01']
by_month={x['month']:x for x in values};diffs=[]
for x in values[1:]:
 year,month=map(int,x['month'].split('-'));prev=f'{year-1}-12' if month==1 else f'{year}-{month-1:02d}'
 if prev in by_month:
  delta=x['same_release_prior']-by_month[prev]['current']
  if delta:diffs.append({'current_release_month':x['month'],'prior_reference_month':prev,'prior_same_release':x['same_release_prior'],'preceding_release_current':by_month[prev]['current'],'revision_difference_million_usd':delta})
full_notice=[x for x in notice_hits if x['phrase'] in ('Changes to End-Use Commodity Classification','semiconductor media','reassigned')]
result={'status':'FT900_188_month_original_PDF_notice_scan_Jan2012_semiconductor_classification_break','plan_sha256':sha(card/'plan.json'),'PDF_months':188,'all_original_SHA_and_size_pass':True,'same_release_positive_current_prior_pairs':188,'first_five_page_notice_hits':notice_hits,'classification_specific_notice_hits':full_notice,'cross_release_prior_value_differences':diffs,'cross_release_prior_difference_count':len(diffs),'max_abs_cross_release_prior_difference_million_usd':max([abs(x['revision_difference_million_usd']) for x in diffs],default=0),'Jan2012_transition_growth_qualified':False,'Jan2012_transition_reason':'2012-02 original notice explicitly makes semiconductor-media reclassification effective with January 2012 statistics; smart/flash cards move into semiconductors and related devices. December 2011 to January 2012 growth crosses the definition boundary unless a same-basis original bridge is established.','Feb2012_same_release_pair_identity':{'Feb_current':by_month['2012-02']['current'],'Jan_prior_same_release':by_month['2012-02']['same_release_prior'],'Jan_preceding_release_current':by_month['2012-01']['current']},'historical_first_public_bytes_qualified':False,'model_input_qualified':False,'new_external_requests':0,'Qlib_calls':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
print({'status':result['status'],'months':188,'classification_notice_months':sorted(set(x['month'] for x in full_notice)),'cross_release_prior_differences':len(diffs),'Jan2012_transition_qualified':False})

{'status': 'FT900_188_month_original_PDF_notice_scan_Jan2012_semiconductor_classification_break', 'months': 188, 'classification_notice_months': ['2012-02', '2017-01'], 'cross_release_prior_differences': 101, 'Jan2012_transition_qualified': False}


## 中文结论

188/188 份 FT-900 PDF 原始 SHA/尺寸与 Exhibit 7 半导体出口同版当月/上月行核验通过。分类通知位于 2012-02 与 2017-01 月报；对应 2012-01、2017-01 跨定义月增长保留未知。187 组相邻报表有 101 组“同版上月”与前一报表“当月”金额不同，最大差 140 百万美元；未来若构造月变化，只用同份报告两列。当前历史 PDF 首次公开字节与修订时钟仍不合格，不产生 Stage1 因子、拟合或账户。原件视觉审查见 `boundary_adjudication.json`，完整扫描见 `result.json`。SOXX v2 目标未完成。

已有 FT-900 同版月变化 H5 固定信息检验在保留 2012-01 未知的情况下失败；本次发现的 2017-01 一般商品改类是额外的来源限制，不重跑或救援旧版。